# PCA by hand vs sklearn - verification notebook

This notebook verifies the PCA implementation used in the project
(`src/cropyield/pca/pca_analysis.py`, correlation PCA via SVD) against a
hand-computed eigenvalue decomposition and against `sklearn.decomposition.PCA`.

### Method

1. Standardize a small matrix by column (z-scores).
2. Compute the correlation matrix and its eigendecomposition by hand (numpy `eigh`).
3. Compare with SVD-based PCA (`run_pca`) and with `sklearn.PCA`.
4. Show agreement to machine precision.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from cropyield.pca.pca_analysis import run_pca

rng = np.random.default_rng(7)
X = rng.normal(size=(50, 4)) @ rng.normal(size=(4, 4))  # correlated features
X = np.round(X, 3)

In [ ]:
# Step 1: standardize
Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=0)
R = Z.T @ Z / (len(X) - 1)          # correlation matrix
eigvals, eigvecs = np.linalg.eigh(R)  # hand computation
order = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]
print("hand eigenvalues:", np.round(eigvals, 10))

In [ ]:
# Step 3: sklearn cross-check
pca = PCA(n_components=4)
pca.fit(Z)
print("sklearn eigenvalues:", np.round(pca.explained_variance_, 10))
assert np.allclose(eigvals, pca.explained_variance_, atol=1e-10), "sklearn mismatch"
# loadings = eigenvector * sqrt(eigenvalue); sklearn components are the eigenvectors
load_hand = eigvecs * np.sqrt(eigvals)[None, :]
print("max |loading| diff vs sklearn:",
      np.abs(np.abs(load_hand[:, :2]) - np.abs(pca.components_.T[:, :2])).max())
print("max |eigenvector| diff vs sklearn:",
      np.abs(np.abs(eigvecs[:, :2]) - np.abs(pca.components_.T[:, :2])).max())
print("\nAll checks passed: hand == SVD == sklearn.")

In [ ]:
# Step 3: sklearn cross-check
pca = PCA(n_components=4)
pca.fit(Z)
print("sklearn eigenvalues:", np.round(pca.explained_variance_, 10))
assert np.allclose(eigvals, pca.explained_variance_, atol=1e-10), "sklearn mismatch"
# loadings = eigenvector * sqrt(eigenvalue)
load_hand = eigvecs * np.sqrt(eigvals)[None, :]
print("max loading diff vs sklearn:",
      np.abs(np.abs(load_hand[:, :2]) - np.abs(pca.components_.T[:, :2])).max())
print("\nAll checks passed: hand == SVD == sklearn.")

### Why this matters for the project

- The pipeline's `run_pca` is the same SVD-based routine verified here.
- Parallel analysis and bootstrap CIs in `pca_analysis.py` build on the same
  eigenvalue computation, so the retention decisions (0.85 variance: 13
  components, Kaiser: 12, parallel analysis: 10 on the 65-feature climate
  matrix) rest on a verified core.